# NB08 — General Metabolite Source Attribution
**Goal:** Identify microbial candidate producers/consumers for *all* 150 significantly-correlated metabolites (not only the 7 polyamines) using the same 7-stream evidence framework as NB07.

| Stream | Source | Criterion |
|---|---|---|
| E1_shap_stable | `ml_results.pkl` shap_summary | Top-20 SHAP per target AND mean_shap > median + 1 SD |
| E2_spearman | `spearman_correlations_partial.csv` | survives partial adjustment |
| E3_kegg | `module_b3_uniprot_enzymes.csv` | Genus carries biosynthetic EC (polyamine-class only) |
| E4_glasso | `d_glasso_partial_correlation_edges_fixed.csv` | GLASSO partial-correlation edge |
| E5_mofa | `mofa_top_loadings.csv` | Species and metabolite co-load in same MOFA factor (|w|>=0.15) |
| E6_mediation | `bootstrap_mediation_results.csv` | Mediation CI excludes zero |
| E7_within_stage | `within_stage_either_significant.csv` | Sig in Stage_I_II OR Stage_III_IV (FDR<0.10) |

**Dependency:** Run NB07 Cell 3 first to generate `within_stage_either_significant.csv`.

In [1]:
import sys, warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── paths ─────────────────────────────────────────────────────────────────────
NB_DIR    = Path('.').resolve()
INTER_DIR = Path(r'C:\Users\andna\Documents\KI\Results\intermediate')
TABLE_DIR = Path(r'C:\Users\andna\Documents\KI\Results\tables')

sys.path.insert(0, str(NB_DIR))
from utils import (, savefig
    DATASET_PRIMARY,
    load_pickle,
    extract_genus,
)

# ── load preprocessed data ───────────────────────────────────────────────────
pkl        = load_pickle(INTER_DIR / 'preprocessed_data.pkl')
species_df = pkl['species_clr'][DATASET_PRIMARY].copy()
mtb_df     = pkl['mtb_log10'][DATASET_PRIMARY].copy()
meta_df    = pkl['harmonized_meta'][DATASET_PRIMARY].copy()

print(f'Species:     {species_df.shape}')
print(f'Metabolites: {mtb_df.shape}')
print(f'Metadata:    {meta_df.shape}')

# ── polyamine KEGG IDs (for E3 enzyme scope) ─────────────────────────────────
POLYAMINE_KEGG = {'C00134', 'C00315', 'C00378', 'C00986', 'C01672', 'C00077', 'C00062'}

POLYAMINE_EC = {
    'C00134': ['4.1.1.17', '4.1.1.18', '4.1.1.19'],
    'C00315': ['2.5.1.16'],
    'C00378': ['2.5.1.22'],
    'C00986': ['4.1.1.19', '3.5.3.11'],
    'C01672': ['4.1.1.18'],
    'C00077': ['4.1.1.17'],
    'C00062': ['4.1.1.19', '3.5.3.11'],
}

EVIDENCE_COLS = [
    'E1_shap_stable', 'E2_spearman', 'E3_kegg',
    'E4_glasso',      'E5_mofa',     'E6_mediation', 'E7_within_stage',
]

Loaded: C:\Users\andna\Documents\KI\Results\intermediate\preprocessed_data.pkl
Species:     (347, 4392)
Metabolites: (347, 249)
Metadata:    (347, 18)


## Step 1 — Seed Matrix from Full Spearman Correlations
Use `spearman_correlations_all.csv` (14,656 FDR-significant pairs, 150 metabolites). Seed with **top-20 species per metabolite by |rho|** (~3,000 rows).

In [2]:
# ── Load full Spearman correlations ──────────────────────────────────────────
sc_all = pd.read_csv(TABLE_DIR / 'spearman_correlations_all.csv')
print(f'spearman_correlations_all: {len(sc_all):,} rows, '
      f'{sc_all["metabolite"].nunique()} metabolites, {sc_all["species"].nunique()} species')
print(f'FDR<0.05: {(sc_all["qval"]<0.05).sum():,}')
print()

# Normalise metabolite key to KEGG ID only
sc_all['kegg_id'] = sc_all['metabolite'].str.split('_').str[0]

# Top-20 per metabolite by |rho| (FDR<0.05)
sig = sc_all[sc_all['qval'] < 0.05].copy()
sig['abs_rho'] = sig['rho'].abs()
seed_rows = (
    sig.sort_values('abs_rho', ascending=False)
       .groupby('metabolite')
       .head(20)
       .reset_index(drop=True)
)
seed_rows = seed_rows.rename(columns={'species': 'Species', 'metabolite': 'Metabolite'})
seed_rows['genus'] = seed_rows['Species'].apply(extract_genus)

print(f'Seed matrix: {len(seed_rows):,} rows '
      f'({seed_rows["Metabolite"].nunique()} metabolites, '
      f'{seed_rows["Species"].nunique()} species)')
print()
print('Top metabolites by n_seed_pairs:')
print(seed_rows['Metabolite'].value_counts().head(10))

spearman_correlations_all: 14,656 rows, 150 metabolites, 500 species
FDR<0.05: 14,656

Seed matrix: 2,992 rows (150 metabolites, 437 species)

Top metabolites by n_seed_pairs:
Metabolite
C02678_Dodecanedioate         20
C02714_N-Acetylputrescine     20
C05629_3-Phenylpropionate     20
C00134_Putrescine             20
C08277_Sebacate               20
C03283_2,4-Diaminobutyrate    20
C05332_Phenethylamine         20
C00398_Tryptamine             20
C00695_Cholate                20
C00431_5-Aminovalerate        20
Name: count, dtype: int64


## Step 2 — E1: Stable SHAP Signal

In [3]:
# ── Load ml_results.pkl shap_summary ────────────────────────────────────────
with open(INTER_DIR / 'ml_results.pkl', 'rb') as f:
    ml_data = pickle.load(f)

shap_summary = ml_data['shap_summary']   # (49 metabolites x 372 species)
print(f'SHAP summary: {shap_summary.shape}  (rows=metabolites, cols=species)')

# Build shap_stable_set: top-20 per target AND mean_shap > median + 1 SD
shap_stable_set = set()
for target_row in shap_summary.index:
    vals   = shap_summary.loc[target_row]
    kid    = str(target_row).split('_')[0]
    top20  = vals.nlargest(20).index
    thresh = vals.median() + vals.std()
    stable = vals[(vals.index.isin(top20)) & (vals > thresh)].index
    for sp in stable:
        shap_stable_set.add((sp, kid))

print(f'Stable SHAP pairs (top-20 AND > median+1SD): {len(shap_stable_set)}')
print('Sample pairs:')
for sp, kid in list(shap_stable_set)[:5]:
    print(f'  {sp} | {kid}')

SHAP summary: (49, 372)  (rows=metabolites, cols=species)
Stable SHAP pairs (top-20 AND > median+1SD): 881
Sample pairs:
  Onthousia sp900758495 | C09815
  CAG-103 sp900757655 | C00879
  Erwinia mallotivora | C00879
  Caecibacter sp003467125 | C00024
  UMGS693 sp900544555 | C00186


## Step 3 — Load Evidence Sources (E2-E7)

In [4]:
# ── E2: Partial-adjusted Spearman ───────────────────────────────────────────
partial_df = pd.read_csv(TABLE_DIR / 'spearman_correlations_partial.csv')
print(f'Partial correlations: {len(partial_df)} rows')
print(f'  survives_adjustment=True: {partial_df["survives_adjustment"].sum()}')

partial_sig_set = set(
    zip(
        partial_df.loc[partial_df['survives_adjustment'] == True, 'species'],
        partial_df.loc[partial_df['survives_adjustment'] == True, 'metabolite']
            .str.split('_').str[0]
    )
)
print(f'  E2 lookup set: {len(partial_sig_set)} pairs')
print()

# ── E3: KEGG enzyme (polyamine ECs only) ─────────────────────────────────────
enzyme_df = pd.read_csv(TABLE_DIR / 'module_b3_uniprot_enzymes.csv')
genus_ec_map = enzyme_df.groupby('Genus')['EC'].apply(set).to_dict()

def has_kegg_enzyme(species, kegg_id):
    required = POLYAMINE_EC.get(kegg_id, [])
    if not required:
        return False  # not a polyamine — enzyme evidence not mapped
    genus = extract_genus(species).lower()
    return any(ec in genus_ec_map.get(genus, set()) for ec in required)

print(f'E3: enzyme map covers {len(genus_ec_map)} genera')
print()

# ── E4: GLASSO ───────────────────────────────────────────────────────────────
glasso_df = pd.read_csv(TABLE_DIR / 'd_glasso_partial_correlation_edges_fixed.csv')
print(f'GLASSO edges: {len(glasso_df)}')
glasso_sig_set = set(
    zip(glasso_df['Species'], glasso_df['Metabolite'].str.split('_').str[0])
)
print()

# ── E5: MOFA co-loading ──────────────────────────────────────────────────────
mofa_df = pd.read_csv(TABLE_DIR / 'mofa_top_loadings.csv')
MOFA_W = 0.15

# Build: kegg_id -> set of species that co-load in the same factor
mofa_sp_for_mtb = {}
for factor in mofa_df['factor'].unique():
    f_mtbs = mofa_df[
        (mofa_df['factor'] == factor) &
        (mofa_df['view'] == 'metabolite') &
        (mofa_df['abs_weight'] >= MOFA_W)
    ]
    f_sps = set(mofa_df[
        (mofa_df['factor'] == factor) &
        (mofa_df['view'] == 'species') &
        (mofa_df['abs_weight'] >= MOFA_W)
    ]['feature'].values)
    for _, row in f_mtbs.iterrows():
        kid = str(row['feature']).split('_')[0]
        mofa_sp_for_mtb.setdefault(kid, set()).update(f_sps)

print(f'MOFA: {len(mofa_df["factor"].unique())} factors, '
      f'{len(mofa_sp_for_mtb)} metabolites with co-loading species')
for kid, sps in mofa_sp_for_mtb.items():
    print(f'  {kid}: {len(sps)} species')
print()

# ── E6: Bootstrap mediation ──────────────────────────────────────────────────
med_df = pd.read_csv(TABLE_DIR / 'bootstrap_mediation_results.csv')
med_sig = med_df[(med_df['acme_ci_lo'] * med_df['acme_ci_hi']) > 0].copy()
med_sig_set = set(
    zip(med_sig['species'], med_sig['metabolite'].str.split('_').str[0])
)
print(f'Significant mediation pairs: {len(med_sig_set)}')
print()

# ── E7: Within-stage either-sig ──────────────────────────────────────────────
ws_path = TABLE_DIR / 'within_stage_either_significant.csv'
if ws_path.exists():
    ws_df = pd.read_csv(ws_path)
    ws_sig_set = set(zip(ws_df['species'], ws_df['metabolite'].str.split('_').str[0]))
    print(f'Within-stage either-sig pairs: {len(ws_sig_set)}')
else:
    ws_sig_set = set()
    print('WARNING: within_stage_either_significant.csv not found.')
    print('E7 will be 0 until NB07 Cell 3 is re-run.')

Partial correlations: 2000 rows
  survives_adjustment=True: 1997
  E2 lookup set: 1997 pairs

E3: enzyme map covers 30 genera

GLASSO edges: 43

MOFA: 3 factors, 19 metabolites with co-loading species
  C01118: 10 species
  C00954: 10 species
  C06524: 10 species
  C01047: 10 species
  C00440: 10 species
  C00299: 10 species
  C00383: 10 species
  C03570: 10 species
  C05341: 10 species
  C00398: 20 species
  C02714: 10 species
  C00134: 10 species
  C01744: 10 species
  C08277: 10 species
  C00245: 10 species
  C06103: 10 species
  C11284: 10 species
  C06104: 10 species
  C02679: 10 species

Significant mediation pairs: 8

E7 will be 0 until NB07 Cell 3 is re-run.


## Step 4 — Expand Seed with GLASSO, MOFA and Mediation Pairs
These streams may nominate species not captured by the top-20 Spearman filter.

In [5]:
extra_rows = []

# GLASSO edges not in seed
for _, row in glasso_df.iterrows():
    sp  = row['Species']
    mtb = str(row['Metabolite'])
    kid = mtb.split('_')[0]
    if not ((seed_rows['Species'] == sp) & (seed_rows['kegg_id'] == kid)).any():
        extra_rows.append({'Species': sp, 'Metabolite': mtb, 'kegg_id': kid,
                           'genus': extract_genus(sp)})
n_glasso = len(extra_rows)

# MOFA co-loading pairs not in seed
for kid, sp_set in mofa_sp_for_mtb.items():
    mtb_matches = seed_rows[seed_rows['kegg_id'] == kid]['Metabolite']
    mtb_name = mtb_matches.iloc[0] if len(mtb_matches) else kid
    for sp in sp_set:
        in_seed  = ((seed_rows['Species'] == sp) & (seed_rows['kegg_id'] == kid)).any()
        in_extra = any(r['Species'] == sp and r['kegg_id'] == kid for r in extra_rows)
        if not in_seed and not in_extra:
            extra_rows.append({'Species': sp, 'Metabolite': mtb_name, 'kegg_id': kid,
                               'genus': extract_genus(sp)})
n_mofa = len(extra_rows) - n_glasso

# Mediation pairs not in seed
for _, row in med_sig.iterrows():
    sp  = row['species']
    mtb = str(row['metabolite'])
    kid = mtb.split('_')[0]
    in_seed  = ((seed_rows['Species'] == sp) & (seed_rows['kegg_id'] == kid)).any()
    in_extra = any(r['Species'] == sp and r['kegg_id'] == kid for r in extra_rows)
    if not in_seed and not in_extra:
        extra_rows.append({'Species': sp, 'Metabolite': mtb, 'kegg_id': kid,
                           'genus': extract_genus(sp)})
n_med = len(extra_rows) - n_glasso - n_mofa

print(f'Extra pairs: {len(extra_rows)}  (GLASSO: {n_glasso}, MOFA: {n_mofa}, Mediation: {n_med})')

if extra_rows:
    ev_matrix = pd.concat(
        [seed_rows[['Species','Metabolite','kegg_id','genus']],
         pd.DataFrame(extra_rows)],
        ignore_index=True
    )
else:
    ev_matrix = seed_rows[['Species','Metabolite','kegg_id','genus']].copy()

for col in EVIDENCE_COLS:
    ev_matrix[col] = False

print(f'Final evidence matrix: {len(ev_matrix):,} rows '
      f'({ev_matrix["Metabolite"].nunique()} metabolites, '
      f'{ev_matrix["Species"].nunique()} species)')

Extra pairs: 192  (GLASSO: 40, MOFA: 152, Mediation: 0)
Final evidence matrix: 3,184 rows (153 metabolites, 440 species)


## Step 5 — Score All 7 Evidence Streams

In [6]:
def score_row(r):
    sp  = r['Species']
    kid = r['kegg_id']
    return pd.Series({
        'E1_shap_stable':  (sp, kid) in shap_stable_set,
        'E2_spearman':     (sp, kid) in partial_sig_set,
        'E3_kegg':         has_kegg_enzyme(sp, kid),
        'E4_glasso':       (sp, kid) in glasso_sig_set,
        'E5_mofa':         sp in mofa_sp_for_mtb.get(kid, set()),
        'E6_mediation':    (sp, kid) in med_sig_set,
        'E7_within_stage': (sp, kid) in ws_sig_set,
    })

ev_matrix[EVIDENCE_COLS] = ev_matrix.apply(score_row, axis=1)

ev_matrix['Evidence_score'] = ev_matrix[EVIDENCE_COLS].sum(axis=1)

def confidence_tier(score):
    if score >= 4: return 'High (4+)'
    if score >= 3: return 'Medium-High (3)'
    if score >= 2: return 'Medium (2)'
    if score >= 1: return 'Low-Medium (1)'
    return 'Insufficient (0)'

ev_matrix['Confidence'] = ev_matrix['Evidence_score'].map(confidence_tier)

# Save
out = ev_matrix.drop(columns=['kegg_id', 'genus'], errors='ignore')
out.to_csv(TABLE_DIR / 'general_metabolite_evidence_matrix.csv', index=False)
print(f'Saved general_metabolite_evidence_matrix.csv ({len(out)} rows)')

print('\nEvidence stream fill rates:')
for col in EVIDENCE_COLS:
    n = ev_matrix[col].sum()
    print(f'  {col}: {n} / {len(ev_matrix)} ({100*n/len(ev_matrix):.1f}%)')

print('\nConfidence tier distribution:')
print(ev_matrix['Confidence'].value_counts())

Saved general_metabolite_evidence_matrix.csv (3184 rows)

Evidence stream fill rates:
  E1_shap_stable: 288 / 3184 (9.0%)
  E2_spearman: 1027 / 3184 (32.3%)
  E3_kegg: 7 / 3184 (0.2%)
  E4_glasso: 43 / 3184 (1.4%)
  E5_mofa: 200 / 3184 (6.3%)
  E6_mediation: 8 / 3184 (0.3%)
  E7_within_stage: 0 / 3184 (0.0%)

Confidence tier distribution:
Confidence
Insufficient (0)    1786
Low-Medium (1)      1238
Medium (2)           148
Medium-High (3)        9
High (4+)              3
Name: count, dtype: int64


## Step 6 — Top Candidates by Metabolite Class

In [7]:
pathway_map = (
    sc_all.drop_duplicates('metabolite')
          .set_index('metabolite')['pathway'].to_dict()
)

top = (
    ev_matrix[ev_matrix['Evidence_score'] > 0]
    .sort_values('Evidence_score', ascending=False)
    .reset_index(drop=True)
)
top['pathway'] = top['Metabolite'].map(pathway_map).fillna('Other')

print(f'Candidates with Evidence_score > 0: {len(top):,}')
print()

cols = ['Species', 'Metabolite'] + EVIDENCE_COLS + ['Evidence_score', 'Confidence']

print('=== Top 30 candidates (all pathways) ===')
print(top.head(30)[cols].to_string(index=False))
print()

print('=== By pathway ===')
for pathway, grp in top.groupby('pathway'):
    print(f'\n{pathway} ({len(grp)} pairs):')
    print(grp.sort_values('Evidence_score', ascending=False)
             .head(10)[cols].to_string(index=False))

top.to_csv(TABLE_DIR / 'general_metabolite_top_candidates.csv', index=False)
print('\nSaved general_metabolite_top_candidates.csv')

Candidates with Evidence_score > 0: 1,398

=== Top 30 candidates (all pathways) ===
                      Species                          Metabolite  E1_shap_stable  E2_spearman  E3_kegg  E4_glasso  E5_mofa  E6_mediation  E7_within_stage  Evidence_score      Confidence
       Aphodousia sp900553105                     C00383_Malonate            True         True    False      False     True          True            False               4       High (4+)
Sediminibacterium sp902168225                     C00383_Malonate            True         True    False      False     True          True            False               4       High (4+)
       Aphodousia sp900553105         C01118_o-Succinylhomoserine            True         True    False      False     True          True            False               4       High (4+)
          UBA1829 sp900548385                     C00383_Malonate            True         True    False      False     True         False            False              

## Step 7 — High-Confidence Candidate Heatmap

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

high_conf = top[top['Evidence_score'] >= 2].copy()
print(f'Medium-confidence or above (score>=2): {len(high_conf)}')

if len(high_conf) > 0:
    pivot = (
        high_conf.pivot_table(
            index='Species', columns='Metabolite',
            values='Evidence_score', aggfunc='max', fill_value=0)
        .astype(float)
    )
    # Top 30 species and metabolites
    row_top = pivot.sum(axis=1).nlargest(30).index
    col_top = pivot.sum(axis=0).nlargest(30).index
    pivot_plot = pivot.reindex(index=row_top, columns=col_top, fill_value=0)

    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(pivot_plot, cmap='YlOrRd', linewidths=0.3,
                ax=ax, cbar_kws={'label': 'Evidence score'})
    ax.set_title('General Metabolite Source Attribution\nHigh-Confidence Candidates'
                 ' (top 30 species x top 30 metabolites)', fontsize=11)
    plt.xticks(rotation=45, ha='right', fontsize=7)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    savefig(fig, 'source_attribution', 'nb08_general_metabolite_evidence_heatmap.png', dpi=150)
    plt.close(fig)
    print(f'Saved heatmap -> {out_path}')
else:
    print('No medium-confidence candidates to plot.')

## Step 8 — Novel Candidate Species
Species with strong partial Spearman evidence (E2+) but no prior enzyme annotation (E3-) or mediation (E6-) — purely data-driven candidates.

In [9]:
novel = top[
    (top['E2_spearman'] == True) &
    (top['E3_kegg'] == False) &
    (top['E6_mediation'] == False)
].copy()

novel = novel.merge(
    sc_all.rename(columns={'species':'Species','metabolite':'Metabolite'})
          [['Species','Metabolite','rho']].drop_duplicates(),
    on=['Species','Metabolite'], how='left'
)
novel['abs_rho'] = novel['rho'].abs()

print(f'Novel candidates (E2+ only, no prior annotation): {len(novel)}')
print()
print('Top 30 novel candidates sorted by evidence_score then |rho|:')
print(
    novel.sort_values(['Evidence_score','abs_rho'], ascending=[False,False])
         .head(30)
         [['Species','Metabolite','pathway','rho'] + EVIDENCE_COLS + ['Evidence_score']]
         .to_string(index=False)
)

novel.to_csv(TABLE_DIR / 'general_metabolite_novel_candidates.csv', index=False)
print('\nSaved general_metabolite_novel_candidates.csv')

Novel candidates (E2+ only, no prior annotation): 1018

Top 30 novel candidates sorted by evidence_score then |rho|:
                       Species                  Metabolite             pathway       rho  E1_shap_stable  E2_spearman  E3_kegg  E4_glasso  E5_mofa  E6_mediation  E7_within_stage  Evidence_score
           UMGS687 sp900545735 C01118_o-Succinylhomoserine               Other -0.342032            True         True    False      False     True         False            False               3
           UMGS687 sp900545735             C00383_Malonate               Other -0.317715            True         True    False      False     True         False            False               3
 Sediminibacterium sp902168225 C01118_o-Succinylhomoserine               Other -0.296964            True         True    False      False     True         False            False               3
           UBA1829 sp900548385             C00383_Malonate               Other -0.243046            True   

## Step 9 — Stage-Stratified Dominant Stage Lookup
For each (species, metabolite) pair, identify the **dominant stage** (min q-value across Healthy / Early_CRC / Advanced_CRC) and compute −log₁₀(q) for bubble sizing in Steps 10–12.

In [14]:
import pickle

STAGE_COLORS = {
    "Healthy":      "#2ca02c",   # green
    "Early_CRC":    "#f5c518",   # amber
    "Advanced_CRC": "#7b2d8b",   # purple
}
THREE_GROUP_ORDER_LOCAL = ["Healthy", "Early_CRC", "Advanced_CRC"]
STAGE_N = {"Healthy": 127, "Early_CRC": 57, "Advanced_CRC": 163}  # sample counts

with open(INTER_DIR / 'analysis_results.pkl', 'rb') as _f:
    _ar = pickle.load(_f)
strat_corr = _ar['strat_corr']

print("strat_corr stages:", list(strat_corr.keys()))
for _g, _df in strat_corr.items():
    print(f"  {_g}: {len(_df):,} pairs  (n={STAGE_N.get(_g, '?')} samples)")

_frames = []
for stage in THREE_GROUP_ORDER_LOCAL:
    if stage not in strat_corr:
        print(f"  WARNING: stage '{stage}' not found in strat_corr — skipping")
        continue
    _df = strat_corr[stage][['species', 'metabolite', 'rho', 'qval']].copy()
    _df['stage'] = stage
    _frames.append(_df)

strat_long = pd.concat(_frames, ignore_index=True).dropna(subset=['qval'])
strat_long['neg_log10_qval'] = (
    -np.log10(strat_long['qval'].clip(lower=1e-300))
).clip(lower=0)
strat_long['abs_rho'] = strat_long['rho'].abs()

# Dominant stage = stage with the largest |rho|.
# Using abs_rho (not qval) because q-values are computed in separate per-stage BH
# pools of different sizes and are therefore not directly comparable across stages.
dom_idx = strat_long.groupby(['species', 'metabolite'])['abs_rho'].idxmax()
dom_stage_df = (
    strat_long.loc[dom_idx,
                   ['species', 'metabolite', 'stage', 'neg_log10_qval', 'rho', 'abs_rho']]
    .rename(columns={'stage': 'dominant_stage'})
    .set_index(['species', 'metabolite'])
)

print(f"\ndom_stage_df: {len(dom_stage_df):,} pairs")
print("Dominant stage counts:")
print(dom_stage_df['dominant_stage'].value_counts())
print(f"\nabs_rho range: "
      f"{dom_stage_df['abs_rho'].min():.3f} – {dom_stage_df['abs_rho'].max():.3f}")
print(f"neg_log10_qval range: "
      f"{dom_stage_df['neg_log10_qval'].min():.2f} – {dom_stage_df['neg_log10_qval'].max():.2f}")


strat_corr stages: ['Healthy', 'Early_CRC', 'Advanced_CRC']
  Healthy: 2,689 pairs  (n=127 samples)
  Early_CRC: 310 pairs  (n=57 samples)
  Advanced_CRC: 2,939 pairs  (n=163 samples)

dom_stage_df: 5,024 pairs
Dominant stage counts:
dominant_stage
Healthy         2386
Advanced_CRC    2332
Early_CRC        306
Name: count, dtype: int64

abs_rho range: 0.214 – 0.680
neg_log10_qval range: 1.30 – 10.26


## Step 10 — Full All-vs-All Association Starmap
Navy-background heatmap of **all stage-stratified FDR-significant species–metabolite pairs** (union across Healthy / Early_CRC / Advanced_CRC). Background cells: dominant-stage |Spearman ρ|. Each dot = one pair; colour = dominant disease stage (assigned by highest |Spearman ρ|, which is directly comparable across stages); size = −log₁₀(q) of that dominant-stage correlation. Stage legend includes sample size per group (n=127 / 57 / 163).

In [ ]:
import matplotlib.patches as mpatches

def compute_max_bubble_area_pts2(ax, n_rows, n_cols):
    """Area in pts² for a circle with radius = half the smaller cell dimension."""
    fig = ax.figure
    fig.canvas.draw()
    bbox = ax.get_window_extent()
    dpi  = fig.dpi
    cell_h_pts = (bbox.height / dpi * 72) / n_rows
    cell_w_pts = (bbox.width  / dpi * 72) / n_cols
    return np.pi * (min(cell_h_pts, cell_w_pts) / 2) ** 2

# Build pivot from stage-stratified pairs (dom_stage_df).
# Background encodes the dominant-stage |rho|; bubbles are the same pair set.
# This fixes the previous mismatch where the background used the global (n=347)
# correlation (14,656 pairs) while bubbles came from per-stage correlations (fewer pairs).
sc_pivot = dom_stage_df.reset_index().pivot_table(
    index='species', columns='metabolite',
    values='abs_rho', aggfunc='max',
    fill_value=0
)
sp_order  = sc_pivot.sum(axis=1).sort_values(ascending=False).index
mtb_order = sc_pivot.sum(axis=0).sort_values(ascending=False).index
sc_pivot  = sc_pivot.loc[sp_order, mtb_order]
n_sp, n_mtb = sc_pivot.shape
print(f"Stratified pivot: {n_sp} species × {n_mtb} metabolites ({len(dom_stage_df):,} pairs)")

sp_idx  = {s: i for i, s in enumerate(sc_pivot.index)}
mtb_idx = {m: j for j, m in enumerate(sc_pivot.columns)}

# Draw heatmap
fig, ax = plt.subplots(figsize=(40, 30))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#1a1a2e')

sns.heatmap(
    sc_pivot, cmap='Greys', vmin=0, vmax=0.8,
    ax=ax, cbar=False,
    xticklabels=False, yticklabels=False,
    linewidths=0, linecolor='none'
)

max_area = compute_max_bubble_area_pts2(ax, n_sp, n_mtb)

# Draw stage bubbles from dom_stage_df
_bub = dom_stage_df.reset_index().copy()
_bub['i'] = _bub['species'].map(sp_idx)
_bub['j'] = _bub['metabolite'].map(mtb_idx)
_bub = _bub.dropna(subset=['i', 'j'])
_bub[['i', 'j']] = _bub[['i', 'j']].astype(int)
_bub['bubble_size'] = (_bub['neg_log10_qval'] / 4.0).clip(0, 1) * max_area

for stage in THREE_GROUP_ORDER_LOCAL:
    _s = _bub[_bub['dominant_stage'] == stage]
    ax.scatter(
        _s['j'] + 0.5, _s['i'] + 0.5,
        s=_s['bubble_size'], c=STAGE_COLORS[stage],
        alpha=0.75, linewidths=0, label=stage, zorder=3
    )

# Legend — stage labels include sample sizes
stage_handles = [
    mpatches.Patch(
        facecolor=STAGE_COLORS[s],
        label=f"{s.replace('_', ' ')} (n={STAGE_N[s]})"
    )
    for s in THREE_GROUP_ORDER_LOCAL
]
size_handles = []
for ref_val, ref_label in [(1.0, 'q=0.1'), (2.0, 'q=0.01'), (4.0, 'q=1e-4')]:
    ref_s = max(0.0, min(1.0, ref_val / 4.0)) * max_area
    size_handles.append(
        ax.scatter([], [], s=ref_s, c='white', alpha=0.85,
                   label=f'−log₁₀(q)={ref_val:.0f} ({ref_label})')
    )
leg = ax.legend(
    handles=stage_handles + size_handles,
    loc='lower right', framealpha=0.3,
    facecolor='#1a1a2e', edgecolor='white',
    labelcolor='white', fontsize=10,
    title='Stage / Significance', title_fontsize=11
)
leg.get_title().set_color('white')
ax.set_title(
    f'Stage-Stratified Species–Metabolite Associations ({len(dom_stage_df):,} pairs)\n'
    'Background: dominant-stage |Spearman ρ| · Dots: dominant stage (by |ρ|) · Size: −log₁₀(q)',
    fontsize=16, color='white', pad=18
)

plt.tight_layout()
savefig(fig, 'source_attribution', 'nb08_full_association_starmap.png', dpi=150, facecolor='#1a1a2e')
plt.close(fig)
print(f'Saved starmap → {out_path}')


## Step 11 — SHAP Post-ML Heatmap with Stage Bubble Overlay
Top 50 species × 49 ML-covered metabolites. Background: SHAP absolute importance (YlOrRd). Bubbles encode the dominant disease stage (colour) and −log₁₀(q) significance (size).

In [ ]:
# ── Sort and subset SHAP matrix ───────────────────────────────────────────────
sp_order_shap  = shap_summary.sum(axis=0).sort_values(ascending=False).index[:50]
mtb_order_shap = shap_summary.sum(axis=1).sort_values(ascending=False).index
shap_plot      = shap_summary.loc[mtb_order_shap, sp_order_shap].copy()
n_mtb_s, n_sp_s = shap_plot.shape
print(f"SHAP plot: {n_mtb_s} metabolites × {n_sp_s} species")

fig, ax = plt.subplots(figsize=(20, 15))
sns.heatmap(
    shap_plot, cmap='YlOrRd', ax=ax,
    cbar_kws={'label': 'Mean |SHAP|', 'shrink': 0.6},
    xticklabels=True, yticklabels=True,
    linewidths=0.2, linecolor='#cccccc'
)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=6, rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=6, rotation=0)
ax.set_title(
    'SHAP Importance — Top 50 Species × 49 ML Metabolites\n'
    'Bubbles: dominant disease stage · Size: −log₁₀(q)',
    fontsize=11
)

max_area_s = compute_max_bubble_area_pts2(ax, n_mtb_s, n_sp_s)

sp_col_idx  = {sp: j for j, sp in enumerate(shap_plot.columns)}
mtb_row_idx = {mt: i for i, mt in enumerate(shap_plot.index)}

# Build pairs where SHAP > 0 and a stage-sig entry exists
_pairs_s = pd.DataFrame(
    [(mt, sp) for mt in shap_plot.index
               for sp in shap_plot.columns
               if shap_plot.at[mt, sp] > 0],
    columns=['metabolite', 'species']
)
_pairs_s = (
    _pairs_s
    .merge(dom_stage_df.reset_index(), on=['species', 'metabolite'], how='left')
    .dropna(subset=['dominant_stage'])
)
print(f"Stage-matched SHAP pairs: {len(_pairs_s)}")

for stage in THREE_GROUP_ORDER_LOCAL:
    _s    = _pairs_s[_pairs_s['dominant_stage'] == stage]
    js    = _s['species'].map(sp_col_idx)
    is_   = _s['metabolite'].map(mtb_row_idx)
    mask  = js.notna() & is_.notna()
    sizes = (_s.loc[mask, 'neg_log10_qval'] / 4.0).clip(0, 1) * max_area_s
    if mask.sum() > 0:
        ax.scatter(
            js[mask] + 0.5, is_[mask] + 0.5,
            s=sizes, c=STAGE_COLORS[stage], alpha=0.80,
            linewidths=0.3, edgecolors='white',
            label=stage.replace('_', ' '), zorder=4
        )

# Legend
stage_handles_s = [
    mpatches.Patch(facecolor=STAGE_COLORS[s], label=f"{s.replace('_', ' ')} (n={STAGE_N[s]})")
    for s in THREE_GROUP_ORDER_LOCAL
]
size_handles_s = []
for ref_val, ref_label in [(1.0, 'q=0.1'), (2.0, 'q=0.01'), (4.0, 'q=1e-4')]:
    ref_s = max(0.0, min(1.0, ref_val / 4.0)) * max_area_s
    size_handles_s.append(
        ax.scatter([], [], s=ref_s, c='grey', alpha=0.85,
                   label=f'−log₁₀(q)={ref_val:.0f} ({ref_label})')
    )
ax.legend(
    handles=stage_handles_s + size_handles_s,
    loc='upper right', fontsize=7,
    framealpha=0.7, title='Stage / Significance', title_fontsize=8
)

plt.tight_layout()
savefig(fig, 'source_attribution', 'nb08_shap_stage_bubble_heatmap.png', dpi=150)
plt.close(fig)
print(f'Saved SHAP heatmap → {out_path}')


## Step 12 — Directional Spearman Heatmap with Stage Bubbles (Producer / Reducer)
Filters to **evidence_score ≥ 2** pairs. Background: signed Spearman ρ — red = positive (producer signal), blue = negative (reducer/consumer signal). Species rows hierarchically clustered (Ward); metabolite columns sorted by pathway then mean ρ, with pathway boundary lines.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

# ── Filter sc_all to evidence_score ≥ 2 pairs ──────────────────────────────
high_conf_pairs = set(zip(high_conf['Species'], high_conf['Metabolite']))
sc_hc = sc_all[
    sc_all.apply(
        lambda r: (r['species'], r['metabolite']) in high_conf_pairs, axis=1
    )
].copy()
print(f"sc_all pairs with evidence_score ≥ 2: {len(sc_hc)}")

# ── Build signed rho pivot ─────────────────────────────────────────────────
rho_pivot = sc_hc.pivot_table(
    index='species', columns='metabolite',
    values='rho', aggfunc='mean', fill_value=0.0
)

# Sort metabolite columns: by pathway, then by mean rho within pathway
pw_map = sc_all.drop_duplicates('metabolite').set_index('metabolite')['pathway'].to_dict()
col_df = pd.DataFrame({
    'pathway':  [pw_map.get(m, 'Other') for m in rho_pivot.columns],
    'mean_rho': rho_pivot.mean(axis=0).values
}, index=rho_pivot.columns)
col_order = col_df.sort_values(['pathway', 'mean_rho'], ascending=[True, False]).index
rho_pivot = rho_pivot[col_order]

# Ward hierarchical clustering on species rows
_rho_arr  = rho_pivot.fillna(0.0).values
link_mat  = linkage(_rho_arr, method='ward', metric='euclidean')
row_order = dendrogram(link_mat, no_plot=True)['leaves']
rho_pivot = rho_pivot.iloc[row_order]
n_sp_d, n_mtb_d = rho_pivot.shape
print(f"Directional pivot: {n_sp_d} species × {n_mtb_d} metabolites")

# ── Draw heatmap ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(
    rho_pivot, cmap='RdBu_r', center=0, vmin=-0.7, vmax=0.7,
    ax=ax, cbar_kws={'label': 'Spearman ρ', 'shrink': 0.5},
    xticklabels=True, yticklabels=True,
    linewidths=0.3, linecolor='#eeeeee'
)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=6, rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), fontsize=6, rotation=0)
ax.set_title(
    'Directional Spearman ρ — Evidence Score ≥ 2 Pairs\n'
    'Red = producer signal (ρ > 0) · Blue = reducer/consumer (ρ < 0)\n'
    'Rows: species (Ward clustering) · Columns: metabolite (by pathway)',
    fontsize=10
)

# Pathway boundary lines
_prev_pw = None
for _j, _mtb in enumerate(rho_pivot.columns):
    _pw = pw_map.get(_mtb, 'Other')
    if _pw != _prev_pw and _prev_pw is not None:
        ax.axvline(_j, color='black', lw=1.2, zorder=5)
    _prev_pw = _pw

max_area_d = compute_max_bubble_area_pts2(ax, n_sp_d, n_mtb_d)

# Vectorised bubble prep
sp_row_d  = {sp: i for i, sp in enumerate(rho_pivot.index)}
mtb_col_d = {mt: j for j, mt in enumerate(rho_pivot.columns)}
_pairs_d = pd.DataFrame(
    [(sp, mt) for sp in rho_pivot.index
               for mt in rho_pivot.columns
               if rho_pivot.at[sp, mt] != 0.0],
    columns=['species', 'metabolite']
)
_pairs_d = (
    _pairs_d
    .merge(dom_stage_df.reset_index(), on=['species', 'metabolite'], how='left')
    .dropna(subset=['dominant_stage'])
)
print(f"Stage-matched directional pairs: {len(_pairs_d)}")

for stage in THREE_GROUP_ORDER_LOCAL:
    _s    = _pairs_d[_pairs_d['dominant_stage'] == stage]
    js    = _s['metabolite'].map(mtb_col_d)
    is_   = _s['species'].map(sp_row_d)
    mask  = js.notna() & is_.notna()
    sizes = (_s.loc[mask, 'neg_log10_qval'] / 4.0).clip(0, 1) * max_area_d
    if mask.sum() > 0:
        ax.scatter(
            js[mask] + 0.5, is_[mask] + 0.5,
            s=sizes, c=STAGE_COLORS[stage], alpha=0.80,
            linewidths=0.4, edgecolors='white',
            label=stage.replace('_', ' '), zorder=4
        )

# Legend
stage_handles_d = [
    mpatches.Patch(facecolor=STAGE_COLORS[s], label=f"{s.replace('_', ' ')} (n={STAGE_N[s]})")
    for s in THREE_GROUP_ORDER_LOCAL
]
size_handles_d = []
for ref_val, ref_label in [(1.0, 'q=0.1'), (2.0, 'q=0.01'), (4.0, 'q=1e-4')]:
    ref_s = max(0.0, min(1.0, ref_val / 4.0)) * max_area_d
    size_handles_d.append(
        ax.scatter([], [], s=ref_s, c='grey', alpha=0.85,
                   label=f'−log₁₀(q)={ref_val:.0f} ({ref_label})')
    )
ax.legend(
    handles=stage_handles_d + size_handles_d,
    loc='upper right', fontsize=7,
    framealpha=0.7, title='Stage / Significance', title_fontsize=8
)

plt.tight_layout()
savefig(fig, 'source_attribution', 'nb08_directional_stage_bubble_heatmap.png', dpi=150)
plt.close(fig)
print(f'Saved directional heatmap → {out_path}')
